# Pipeline explorer

Run one document through the pipeline and watch each stage. Change the settings
in the next cell, then **Run All**.

Every stage shows what it actually produced as a table, followed by a note on what
that stage did and what tends to go wrong in it.


## Settings

Edit these lines. Everything below follows from them.

| setting | options |
|---|---|
| `SAMPLE` | `1`–`10` for a document from the evaluation set, or `None` to use `PDF_PATH` |
| `PDF_PATH` | path to any PDF — used only when `SAMPLE = None` |
| `MODEL` | `'llama3.1:8b'`, `'gemma4:e4b'`, `'llama3.3:70b'` |
| `EXTRACTOR` | `'pdfplumber'`, `'docling'`, `'lighton'` |
| `RERUN` | `False` reuses saved output; `True` calls the model again |

> With `RERUN = False` this opens instantly if the combination has been run
> before. Set it to `True` after changing the prompt, or the notebook will show
> you output from the old one.


In [ ]:
SAMPLE    = 1                  # 1-10, or None to use PDF_PATH
PDF_PATH  = 'data/input/pdfs/sample1.pdf'
MODEL     = 'llama3.1:8b'
EXTRACTOR = 'pdfplumber'
RERUN     = False


In [ ]:
import json
import os
import time
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

import pandas as pd
from IPython.display import display

from dmpbridge.core import paths as P
from dmpbridge.core.pipeline import run_and_save
from dmpbridge.strategies.wholedoc import WholeDocStrategy

pd.set_option('display.max_colwidth', 78)
pd.set_option('display.width', 180)

# ── Table colours ────────────────────────────────────────────────
# One tint per label, reused in every table below, so a colour means the
# same thing throughout. Text colour is set alongside each background so
# the cells stay readable when the notebook is viewed in a dark theme.
INK = '#111111'
LABEL_TINT = {
    'title':               '#e3edfa',   # blue
    'section.title':       '#fce8df',   # orange
    'section.description': '#ddf3ec',   # green
    'question.text':       '#ece3fa',   # purple
    'answer.text':         '#eceef0',   # grey
}
WARN, GOOD = '#fdecea', '#e6f6ee'


def tint_label(v):
    """Background for a cell holding a label name."""
    c = LABEL_TINT.get(v)
    return f'background-color: {c}; color: {INK}' if c else ''


def flag_empty(v):
    """Mark a cell that should hold text but does not."""
    return f'background-color: {WARN}; color: {INK}' if v == '' else ''


def highlight_row(name, tint=GOOD):
    """Row-wise styler that tints one named row — for the headline metric."""
    return lambda row: [f'background-color: {tint}; color: {INK}'
                        if row.name == name else '' for _ in row]

TAG = P.make_tag(MODEL, EXTRACTOR)

if SAMPLE is not None:
    pdf   = Path(f'data/input/pdfs/sample{SAMPLE}.pdf')
    stem  = f'sample{SAMPLE}'
    paths = {2: P.labeled_path(TAG, SAMPLE), 3: P.structured_path(TAG, SAMPLE),
             4: P.final_path(TAG, SAMPLE)}
else:
    # A PDF from outside the evaluation set: keep its output out of the
    # tagged folders, which are reserved for scored runs.
    pdf   = Path(PDF_PATH)
    stem  = pdf.stem
    scratch = P.OUTPUT_ROOT / 'explorer' / TAG
    paths = {n: scratch / f'{stem}__stage{n}.json' for n in (2, 3, 4)}

if not pdf.exists():
    raise FileNotFoundError(f'No PDF at {pdf}')

display(pd.DataFrame([['document', str(pdf)], ['model', MODEL],
                      ['extractor', EXTRACTOR], ['tag', TAG]],
                     columns=['setting', 'value']).set_index('setting'))

missing = [n for n, p in paths.items() if not p.exists()]
if RERUN or missing:
    why = 'RERUN = True' if RERUN else f'stage(s) {missing} not yet produced'
    print(f'Running the pipeline ({why}) — this calls {MODEL}.')
    t0 = time.time()
    strategy = WholeDocStrategy(model=MODEL, extractor=EXTRACTOR,
                                cache_dir=P.EXTRACTED_DIR / EXTRACTOR)
    run_and_save(strategy, pdf, paths[2], struct_path=paths[3], final_path=paths[4])
    print(f'done in {time.time() - t0:.1f} s')
else:
    print('Using saved output. Set RERUN = True to call the model again.')


## Stage 1 — read the PDF

The extractor turns pages into text blocks. Nothing is classified yet; this step
has no idea what a question or an answer is.


In [ ]:
raw = json.loads((P.EXTRACTED_DIR / EXTRACTOR / f'{stem}.json').read_text(encoding='utf-8'))
blocks1 = raw if isinstance(raw, list) else raw.get('blocks', [])

df1 = pd.DataFrame([{
    'page':  b.get('page'),
    'bold':  b.get('bold'),
    'chars': len(b.get('text', '')),
    'text':  b.get('text', ''),
} for b in blocks1])
df1.index.name = 'block'

print(f'{len(df1)} blocks, {df1["chars"].sum()} characters')

# Length is shaded because short blocks are the interesting ones here —
# a 6-character block is usually a fragment of a wrapped line.
display(df1.head(12).style
        .background_gradient(cmap='Blues', subset=['chars'])
        .format({'chars': '{:d}'}))


**What just happened.** The PDF became a flat list of text blocks, each carrying
position and font information. The model never sees the PDF itself — only this
list, so anything lost here is lost for good.

**What goes wrong here.** `pdfplumber` reads the PDF text layer line by line, so a
wrapped paragraph arrives as several blocks, each of which the model must label on
its own. `docling` and `lighton` segment by paragraph instead.

This stage is cached per *extractor* and shared by every model, which is why
changing `MODEL` above does not re-extract.


## Stage 2 — label every block

The whole document goes to the model in **one call**, and it returns a label and a
confidence for each block.


In [ ]:
raw2 = json.loads(paths[2].read_text(encoding='utf-8'))
blocks2 = raw2 if isinstance(raw2, list) else raw2.get('blocks', [])

df2 = pd.DataFrame([{
    'label':      b.get('label'),
    'confidence': b.get('confidence'),
    'text':       b.get('text', ''),
} for b in blocks2])
df2.index.name = 'block'

counts = df2['label'].value_counts().rename('blocks').to_frame()
display(counts.style
        .map(tint_label, subset=None)
        .apply(lambda col: [tint_label(i) for i in counts.index], subset=['blocks']))

display(df2.head(12).style
        .map(tint_label, subset=['label'])
        .format({'confidence': '{:.2f}'}))


**What just happened.** Each block now has one of five labels — `title`,
`section.title`, `section.description`, `question.text`, `answer.text` — plus the
model's own confidence. The definitions come from
[`dmpbridge/prompts/system.py`](../dmpbridge/prompts/system.py).

**What goes wrong here.** Almost every pipeline error starts in this stage, and the
hardest boundary is `question.text` versus `section.title`: a lettered sub-item
(`A.`, `B.`, `C.`) under a heading is a *question*, but it looks like a heading.
Compare the counts above against what the document really contains — if a document
with nine real questions shows only two, they have already been lost here.

Confidence is the model's own claim and is close to `1.00` almost everywhere,
including on wrong labels. Do not read it as reliability.


## Stage 3 — build the structure

The flat list becomes a nested document: sections, each holding questions, each
holding an answer. No model call — this is deterministic, driven entirely by the
labels from stage 2.


In [ ]:
def unpack(path):
    """(title, sections) from the DMP-tool narrative schema."""
    tpl = json.loads(path.read_text(encoding='utf-8')).get('narrative', {}).get('template', {})
    return tpl.get('title', ''), tpl.get('section', [])


def to_frame(sections):
    """One row per question, carrying its section."""
    rows = []
    for si, s in enumerate(sections, 1):
        qs = s.get('question', [])
        if not qs:
            rows.append({'section': si, 'section title': s.get('title', ''),
                         'question': '', 'answer': ''})
        for q in qs:
            ans = (q.get('answer') or {}).get('json', {}).get('answer', '')
            rows.append({'section': si, 'section title': s.get('title', ''),
                         'question': str(q.get('text', '')), 'answer': str(ans)})
    return pd.DataFrame(rows)


title3, sections3 = unpack(paths[3])
df3 = to_frame(sections3)
print(f'title: {title3!r}')
print(f'{len(sections3)} sections, {df3["question"].ne("").sum()} questions with text')

# A blank question is flagged: the section exists but nothing was put in it,
# which is what a mislabelled heading leaves behind.
display(df3.head(12).style.map(flag_empty, subset=['question', 'answer']))


**What just happened.** Consecutive blocks sharing a label were merged, and the
hierarchy rebuilt: every `section.title` opens a new section, every `question.text`
opens a question inside it, and following `answer.text` becomes that question's
answer. This is the shape the DMP Tool expects.

**What goes wrong here.** Nothing is invented at this stage, but stage 2's mistakes
change shape. A question mislabelled as a heading does not merely lose one label —
it **opens a whole new section**, so a single wrong label restructures the document.
That is why the section count can be far higher than the document really has, and
why rows above may show a section title with an empty question beneath it.


## Stage 4 — apply the annotation rules

A deterministic pass from `data/input/Rules.xlsx`: where a question has no text, it
is filled from the section heading, then the section description, then the document
title — whichever exists first.


In [ ]:
title4, sections4 = unpack(paths[4])
df4 = to_frame(sections4)

display(pd.DataFrame([
    {'sections': len(sections3), 'questions with text': int(df3['question'].ne('').sum())},
    {'sections': len(sections4), 'questions with text': int(df4['question'].ne('').sum())},
], index=['stage 3', 'stage 4 (rules applied)']))

changed = pd.DataFrame({
    'was': df3['question'],
    'now': df4['question'],
    'filled from': df4['section title'],
})
changed = changed[changed['was'] != changed['now']]
print(f'{len(changed)} question(s) filled in by the rules')
if len(changed):
    display(changed.head(10).style
            .map(flag_empty, subset=['was'])
            .map(lambda v: f'background-color: {GOOD}; color: {INK}' if v else '',
                 subset=['now']))


**What just happened.** Stage 3 is kept unconverted and stage 4 written beside it,
so the two can be compared — which is what the table above does.

**What goes wrong here.** The rules only fill questions that are *empty*. They cannot
repair a question mislabelled as a heading in stage 2, because that question is not
blank — it is missing entirely, and its text is sitting in a section title instead.

Note what that means when you read the two tables together: a question the model
turned into a heading leaves an empty question inside that new section, which the
rules then fill *from the heading*. The text arrives in roughly the right place by a
completely different route than intended.


## Scoring — the two paths

The pipeline is scored twice, before and after the rules:

| | what is scored | against |
|---|---|---|
| **Path A** | stage 3, as the model produced it | the original annotation |
| **Path B** | stage 4, after the rules ran | the revised annotation |

Path A measures the model alone. Path B measures the model *plus* the deterministic
rules, which is what the pipeline actually delivers.

The two use **different reference versions**, so the gap between them is the
contribution of the rules — not a second opinion on the same quantity. Supports can
differ too, which is why the counts below may not match.

Only the 10 evaluation samples have annotations; any other PDF cannot be scored.


In [ ]:
def score_path(pred_path, gold_path, dedup):
    """Match one stage's output against one annotation version.

    `dedup` drops a question whose text equals its section title. Path A wants
    that; Path B must not, because the rules fill questions *from* the section
    title, so deduping would discard everything they produce.
    """
    gold = extract_gold(gold_path, dedup_question_title=dedup)
    records, no_gold = _match_structured(pred_path, gold, dedup_question_title=dedup)
    return micro_prf1(_confusion_from_match(records, no_gold)), records, no_gold


def mistakes(records, no_gold):
    """Every wrong label and every block with no counterpart, as a frame."""
    return pd.DataFrame(
        [{'annotation says': r['gold_label'], 'model said': r['pred_label'],
          'text': r['pred_text']}
         for r in records if r['pred_label'] and r['pred_label'] != r['gold_label']]
        + [{'annotation says': 'not in annotation', 'model said': lab, 'text': t}
           for t, lab in no_gold])


if SAMPLE is None:
    print('No reference annotation for a user-supplied PDF — nothing to score.')
    print('The output above is the pipeline result; correctness is for you to judge.')
else:
    from dmpbridge.evaluation.annotation_rules import resolve_new_gt_path
    from dmpbridge.evaluation.evaluate import (
        _confusion_from_match, _match_structured, extract_gold, micro_prf1,
        resolve_old_gt_path,
    )

    a, rec_a, ng_a = score_path(paths[3], resolve_old_gt_path(SAMPLE), True)
    b, rec_b, ng_b = score_path(paths[4], resolve_new_gt_path(SAMPLE), False)

    both = pd.DataFrame(
        [[a['tp'], b['tp']], [a['fp'], b['fp']], [a['fn'], b['fn']],
         [a['precision'], b['precision']], [a['recall'], b['recall']],
         [a['f1'], b['f1']]],
        index=['TP', 'FP', 'FN', 'precision', 'recall', 'f1-score'],
        columns=['Path A  (stage 3)', 'Path B  (stage 4)'])
    display(both.style
            .apply(highlight_row('f1-score'), axis=1)
            .format('{:.3f}'))

    d = b['f1'] - a['f1']
    verdict = 'unchanged' if abs(d) < 0.001 else ('better' if d > 0 else 'worse')
    print(f'The rules made this document {verdict} ({d:+.3f} f1).')
    print('Note the two paths use different annotation versions, so a small '
          'difference\nmay be the reference changing rather than the rules helping.')

    err_a, err_b = mistakes(rec_a, ng_a), mistakes(rec_b, ng_b)

    summary = pd.concat([
        (err_a['annotation says'] + '  ->  ' + err_a['model said'])
        .value_counts().rename('Path A'),
        (err_b['annotation says'] + '  ->  ' + err_b['model said'])
        .value_counts().rename('Path B'),
    ], axis=1).fillna(0).astype(int)
    summary['change'] = summary['Path B'] - summary['Path A']
    print(f'\nMistakes by kind — {len(err_a)} in Path A, {len(err_b)} in Path B:')
    # Diverging scale on `change`: green where the rules removed mistakes,
    # red where they introduced them, white at no change. Symmetric limits,
    # so the same magnitude gets the same intensity either side of zero.
    lim = max(abs(summary['change']).max(), 1)
    display(summary.sort_values('Path A', ascending=False).style
            .background_gradient(cmap='RdYlGn_r', subset=['change'],
                                 vmin=-lim, vmax=lim))


**Reading this.** A block is matched to the annotation by shared words, then judged
on its label. `f1-score` combines precision and recall and stays low unless both are
high, so a model cannot score well by labelling very little, or by labelling
everything it can think of.

**The `change` column is the one to read.** It shows which mistakes the rules
repaired and which they left alone. A rule can only fill a question that is *empty*,
so an error where the model put a question's text into a section heading stays put —
that question is not blank, it is missing.

The per-kind table matters more than the totals. If one row dominates — the same
`question.text -> section.title` repeated — that is a single systematic problem worth
fixing, not many separate accidents.


In [ ]:
if SAMPLE is not None and len(err_a):
    print('Path A — every mistake on this document:')
    # Same label tints as stage 2, so the pair of columns can be read as
    # 'this colour became that colour'.
    display(err_a.style.map(tint_label, subset=['annotation says', 'model said']))


---

**Try next:** change `MODEL` in the settings cell and Run All. Extraction is cached,
so only the labeling re-runs, and any difference you see is the model's alone.
